In [ ]:
!pip install google-generativeai google-genai gspread httpx beautifulsoup4 apify-client google-auth requests pandas


In [ ]:
from __future__ import annotations

import json
import pathlib
import re
from getpass import getpass

try:
    from google.colab import files
except ImportError:
    files = None

ROOT = pathlib.Path('.').resolve()
CONFIG_PATH = ROOT / 'lead_hunter' / 'config.py'

print('Preencha as credenciais. Enter deixa em branco quando quiser pular.')
values = {
    'GOOGLE_MAPS_API_KEY': getpass('Google Maps API key: '),
    'APIFY_TOKEN': getpass('Apify token: '),
    'GEMINI_API_KEY': getpass('Gemini API key: '),
    'GOOGLE_SHEETS_ID': input('Google Sheets ID: ').strip(),
    'NOTIFICATION_EMAIL': input('Email que recebe o resumo: ').strip(),
    'SMTP_EMAIL': input('Gmail remetente: ').strip(),
    'SMTP_APP_PASSWORD': getpass('Gmail App Password: '),
}

if files is not None:
    print('Faça upload do service_account.json da service account do Google Sheets.')
    uploaded = files.upload()
    if uploaded:
        file_name = next(iter(uploaded))
        (ROOT / 'service_account.json').write_bytes(uploaded[file_name])
        print(f'Arquivo salvo como {ROOT / "service_account.json"}')

config_text = CONFIG_PATH.read_text(encoding='utf-8')

def replace_assignment(text: str, variable: str, value: str) -> str:
    pattern = rf'^{variable} = ".*"$'
    replacement = f'{variable} = {json.dumps(value, ensure_ascii=False)}'
    return re.sub(pattern, replacement, text, flags=re.MULTILINE)

for key, value in values.items():
    config_text = replace_assignment(config_text, key, value)

CONFIG_PATH.write_text(config_text, encoding='utf-8')
print('config.py atualizado com sucesso.')
print('Se ainda não compartilhou a planilha com a service account, faça isso antes de rodar o pipeline.')


In [ ]:
from __future__ import annotations

import json
import pathlib

from apify_client import ApifyClient
from google.oauth2.service_account import Credentials
import gspread

from lead_hunter import config

results = {}

try:
    import requests
    response = requests.post(
        config.MAPS_TEXT_SEARCH_URL,
        headers={
            'Content-Type': 'application/json',
            'X-Goog-Api-Key': config.GOOGLE_MAPS_API_KEY,
            'X-Goog-FieldMask': 'places.id,places.displayName',
        },
        json={'textQuery': 'hamburgueria em Pinheiros, São Paulo', 'pageSize': 1, 'languageCode': 'pt-BR', 'regionCode': 'BR'},
        timeout=20,
    )
    results['maps'] = f'OK ({response.status_code})' if response.ok else f'Falhou ({response.status_code})'
except Exception as exc:
    results['maps'] = f'Erro: {exc}'

try:
    if config.APIFY_TOKEN:
        actor = ApifyClient(config.APIFY_TOKEN).actor('apify/instagram-profile-scraper').get()
        results['apify'] = f"OK ({actor.get('title', 'actor acessado')})"
    else:
        results['apify'] = 'Token vazio'
except Exception as exc:
    results['apify'] = f'Erro: {exc}'

try:
    gemini_text = ''
    try:
        from google import genai as google_genai
        client = google_genai.Client(api_key=config.GEMINI_API_KEY)
        gemini_text = client.models.generate_content(model=config.GEMINI_MODEL, contents='Responda apenas OK').text
    except Exception:
        import google.generativeai as google_generativeai
        google_generativeai.configure(api_key=config.GEMINI_API_KEY)
        model = google_generativeai.GenerativeModel(model_name=config.GEMINI_MODEL)
        gemini_text = model.generate_content('Responda apenas OK').text
    results['gemini'] = gemini_text.strip()[:80]
except Exception as exc:
    results['gemini'] = f'Erro: {exc}'

try:
    service_account_file = pathlib.Path('service_account.json')
    credentials = Credentials.from_service_account_file(
        service_account_file,
        scopes=['https://www.googleapis.com/auth/spreadsheets', 'https://www.googleapis.com/auth/drive'],
    )
    client = gspread.authorize(credentials)
    spreadsheet = client.open_by_key(config.GOOGLE_SHEETS_ID)
    results['sheets'] = f"OK ({spreadsheet.title})"
except Exception as exc:
    results['sheets'] = f'Erro: {exc}'

results


In [ ]:
!python main.py


In [ ]:
from __future__ import annotations

import json
import pathlib

import pandas as pd

snapshot_path = pathlib.Path('data/latest_qualified_leads.json')
if not snapshot_path.exists():
    print('Ainda não há snapshot de leads. Rode a célula de execução primeiro.')
else:
    leads = json.loads(snapshot_path.read_text(encoding='utf-8'))
    if not leads:
        print('Nenhum lead qualificado encontrado até agora.')
    else:
        df = pd.DataFrame(leads)
        preview_columns = [
            'status',
            'score',
            'name',
            'neighborhood',
            'city',
            'instagram_username',
            'followers_count',
            'engagement_rate',
            'google_reviews',
            'google_rating',
        ]
        order_map = {'HOT': 0, 'WARM': 1, 'COLD': 2, 'SKIP': 3}
        df['status_order'] = df['status'].map(order_map).fillna(99)
        display(df.sort_values(['status_order', 'score'], ascending=[True, False])[preview_columns].head(10))
        print('Arquivos locais gerados em ./exports/qualified_leads.csv e ./exports/qualified_leads.html')
